In [1]:
import sys
from pathlib import Path

# Locate project root by walking up until .git is found, then make
# config.py importable regardless of Jupyter's working directory.
_root = Path.cwd().resolve()
while not (_root / ".git").exists():
    if _root == _root.parent:
        raise FileNotFoundError("Could not locate project root (.git not found)")
    _root = _root.parent
sys.path.insert(0, str(_root))

from config import RAW_DATASET_PATH, DATA_PROCESSED
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv(RAW_DATASET_PATH)
print(f"Loaded {len(df):,} rows")

Loaded 100,000 rows


In [2]:
# Filter to diabetic cohort

df = df[df['diabetes'] == 1].copy()
print(f"Diabetic cohort: {len(df):,} rows")

Diabetic cohort: 62,232 rows


In [3]:
# Convert admission and discharge dates to datetime
df['admission_date'] = pd.to_datetime( 
    df['admission_date'], 
    dayfirst=True ) 

df['discharge_date'] = pd.to_datetime(
    df['discharge_date'],
    dayfirst=True )

In [4]:
df['admission_date'] = pd.to_datetime(df['admission_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])
df['length_of_stay'] = (df['discharge_date'] - df['admission_date']).dt.days
df = df.drop(columns=['admission_date', 'discharge_date'])
print(f"length_of_stay range: {df['length_of_stay'].min()} to {df['length_of_stay'].max()}")

length_of_stay range: 1 to 14


In [5]:
cohort_missing = df['medications'].isnull().sum()
print(f"Missing medications WITHIN the diabetic cohort: {cohort_missing}")
if cohort_missing == 0:
    print("The 13,878 missing values documented in the thesis fall entirely OUTSIDE the diabetic "
          "cohort. The fillna('Unknown') step below is a defensive no-op for this dataset - keep it "
          "for robustness against future data, but the 'Unknown' category will not appear in the "
          "trained encoder, and the API must not silently invent it at inference time.")

Missing medications WITHIN the diabetic cohort: 0
The 13,878 missing values documented in the thesis fall entirely OUTSIDE the diabetic cohort. The fillna('Unknown') step below is a defensive no-op for this dataset - keep it for robustness against future data, but the 'Unknown' category will not appear in the trained encoder, and the API must not silently invent it at inference time.


In [6]:
df['medications'] = df['medications'].fillna('Unknown')

In [7]:
categorical_cols = ['sex', 'smoking_status', 'alcohol_use', 'primary_diagnosis', 'medications']

unexpected_missing = {c: df[c].isnull().sum() for c in categorical_cols if df[c].isnull().sum() > 0}
if unexpected_missing:
    raise ValueError(
        f"Unexpected missing values found in categorical columns: {unexpected_missing}. "
        "Investigate before encoding - do not let LabelEncoder silently treat NaN as a valid category."
    )
print("No unexpected missing values. Safe to encode.")

No unexpected missing values. Safe to encode.


In [8]:
from config import MODELS_DIR, ENCODERS_PATH
import joblib

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} categories -> {list(le.classes_)}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(encoders, ENCODERS_PATH)
print(f"\nSaved encoders to {ENCODERS_PATH}")

sex: 3 categories -> ['Female', 'Male', 'Other']
smoking_status: 3 categories -> ['Current', 'Former', 'Never']
alcohol_use: 3 categories -> ['Never', 'Occasional', 'Regular']
primary_diagnosis: 2 categories -> ['Diabetes', 'Diabetes;Hypertension']
medications: 20 categories -> ['DPP4 inhibitor', 'DPP4 inhibitor;ACE inhibitor', 'DPP4 inhibitor;Beta blocker', 'DPP4 inhibitor;Calcium channel blocker', 'DPP4 inhibitor;Thiazide', 'Insulin', 'Insulin;ACE inhibitor', 'Insulin;Beta blocker', 'Insulin;Calcium channel blocker', 'Insulin;Thiazide', 'Metformin', 'Metformin;ACE inhibitor', 'Metformin;Beta blocker', 'Metformin;Calcium channel blocker', 'Metformin;Thiazide', 'Sulfonylurea', 'Sulfonylurea;ACE inhibitor', 'Sulfonylurea;Beta blocker', 'Sulfonylurea;Calcium channel blocker', 'Sulfonylurea;Thiazide']

Saved encoders to C:\Users\hamna\Desktop\MediAlert\models\label_encoders.pkl


In [9]:
assert df['diabetes'].nunique() == 1, "Expected diabetes column to be constant after cohort filter"
df = df.drop(columns=['diabetes'])
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Final shape: (62232, 20)
Columns: ['patient_id', 'age', 'sex', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol', 'hdl', 'ldl', 'glucose', 'creatinine', 'hemoglobin', 'wbc', 'smoking_status', 'alcohol_use', 'hypertension', 'primary_diagnosis', 'medications', 'readmission_30', 'length_of_stay']


In [10]:
assert df.isnull().sum().sum() == 0, "Missing values remain after preprocessing!"
print("Verified: no missing values, no silent NaN-as-category encoding.")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
out_path = DATA_PROCESSED / "diabetic_cohort_preprocessed.csv"
df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Verified: no missing values, no silent NaN-as-category encoding.
Saved to C:\Users\hamna\Desktop\MediAlert\data\processed\diabetic_cohort_preprocessed.csv
